# Python Facts & Properties

A quick-reference cheat sheet of core Python facts, properties, and gotchas — the kind of "did you know" statements that explain *why* code behaves the way it does. Each fact is backed by a runnable proof (using `assert`, so a silent cell = confirmed true, plus a `print` showing the result).

## 1. Everything Is an Object

- In Python, **everything is an object** — numbers, strings, functions, classes, modules, even `None` and types themselves.
- Every object has an identity (`id()`), a type (`type()`), and a value.
- Since functions/classes are objects too, they can be assigned to variables, passed as arguments, stored in lists, etc. ("first-class citizens").

In [ ]:
print(type(42), type("hi"), type([1,2]), type(print), type(int), type(None))

# Functions are first-class objects
def greet():
    return "hi"

f = greet          # assign function to a variable
funcs = [greet, print]  # store in a list
assert f() == "hi"
print("Functions can be passed around just like any other value")

# Even a class is just an instance of 'type'
class Foo: pass
print(type(Foo))          # <class 'type'>
print(isinstance(Foo, type))

## 2. Mutability vs Immutability

| Mutable (can change in place) | Immutable (cannot change in place) |
|---|---|
| `list` | `tuple` |
| `dict` | `str` |
| `set` | `int`, `float`, `complex` |
| `bytearray` | `bool` |
| custom classes (by default) | `frozenset` |
| | `bytes` |

- **Lists are mutable, tuples are not** — a tuple's own contents can't be reassigned, though a *mutable object stored inside* a tuple can still be mutated.
- Immutable objects are hashable (usable as dict keys / set members); most mutable objects are not.
- "Changing" an immutable object (e.g. `x = x + 1`, `s = s.upper()`) actually creates a brand-new object and rebinds the name.

In [ ]:
lst = [1, 2, 3]
lst[0] = 99              # OK - lists are mutable
print(lst)

t = (1, 2, 3)
try:
    t[0] = 99            # tuples are immutable
except TypeError as e:
    print("TypeError:", e)

# A tuple can still "change" if it holds a mutable object
t2 = ([1, 2], 3)
t2[0].append(99)         # allowed - we're mutating the LIST inside the tuple, not the tuple itself
print(t2)

# Strings are immutable - every "modification" returns a new string
s = "hello"
s_id = id(s)
s = s.upper()
assert id(s) != s_id
print("String methods return new objects, they never mutate in place")

# Immutable types are hashable -> usable as dict keys / set elements
d = {(1, 2): "point"}    # tuple as a key: OK
try:
    d2 = {[1, 2]: "point"}   # list as a key: fails
except TypeError as e:
    print("TypeError:", e)

## 3. Identity vs Equality (`is` vs `==`)

- `==` compares **values** (calls `__eq__`).
- `is` compares **identity** — whether two names point to the exact same object in memory (compares `id()`).
- `is` is the correct way to check for `None`, `True`, `False` (singletons).
- CPython **caches small integers (-5 to 256)** and string literals that look like identifiers ("interning") as an optimization — so `is` can appear to work for equality by accident. Never rely on this.

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
c = a

assert a == b        # same value
assert not (a is b)  # different objects
assert a is c         # same object

print("a == b:", a == b, "| a is b:", a is b, "| a is c:", a is c)

# Small int caching (CPython implementation detail - do NOT rely on this in real code)
x = 100
y = 100
print("100 is 100:", x is y)   # True in CPython (cached)

x = int("1000")   # built at runtime so the compiler can't silently dedupe the constant
y = int("1000")
print("1000 is 1000 (built at runtime):", x is y)  # False - large ints aren't cached

# Always use 'is' for None
value = None
print(value is None)   # correct
print(value == None)   # works but is NOT the idiomatic/recommended way

## 4. Variable Scope (LEGB) & Closures

- Python resolves names using the **LEGB** rule: **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in.
- A variable assigned anywhere inside a function is treated as local to that function for its *entire* body, unless declared `global` or `nonlocal` — this can cause `UnboundLocalError` even before the assignment line.
- Closures capture variables **by reference**, not by value — a classic gotcha with loops and lambdas ("late binding").
- List/dict/set comprehensions have their **own scope** in Python 3 (the loop variable doesn't leak into the enclosing scope).

In [ ]:
# LEGB
x = "global"
def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print(x)
    inner()
    print(x)
outer()
print(x)

# UnboundLocalError gotcha
count = 10
def broken():
    try:
        print(count)   # Python already knows 'count' is assigned below, so this line fails
        count = 1
    except UnboundLocalError as e:
        print("UnboundLocalError:", e)
broken()

# Late-binding closures: all lambdas share the SAME variable 'i'
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])          # [2, 2, 2] - NOT [0, 1, 2] !

# Fix: capture the value via a default argument
funcs_fixed = [lambda i=i: i for i in range(3)]
print([f() for f in funcs_fixed])    # [0, 1, 2]

# Comprehensions have their own scope (loop var doesn't leak)
[y for y in range(3)]
try:
    print(y)
except NameError as e:
    print("NameError:", e)

## 5. Function Arguments

- Python passes arguments by **"assignment"** (often called "pass by object reference"): the parameter name is bound to the same object as the caller's argument. Mutating a mutable argument in place affects the caller; rebinding the parameter to a new object does not.
- **Mutable default arguments are evaluated once**, at function-definition time, and reused across calls — a very common bug source.
- `*args` collects extra positional args into a tuple; `**kwargs` collects extra keyword args into a dict.
- Parameters can be forced keyword-only (after `*`) or positional-only (before `/`).

In [ ]:
# Pass by object reference
def mutate(lst):
    lst.append(4)          # mutates the caller's list

def rebind(lst):
    lst = [9, 9, 9]         # only rebinds the local name, caller is unaffected

nums = [1, 2, 3]
mutate(nums)
print(nums)                 # [1, 2, 3, 4] - caller sees the mutation

rebind(nums)
print(nums)                 # unchanged - rebinding doesn't escape the function

# Classic mutable-default-argument bug
def append_item(item, bucket=[]):   # DANGER: default list is created ONCE
    bucket.append(item)
    return bucket

print(append_item(1))   # [1]
print(append_item(2))   # [1, 2]  <- surprising! same list reused across calls

# Correct pattern: use None as sentinel
def append_item_fixed(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(append_item_fixed(1))
print(append_item_fixed(2))

# Keyword-only and positional-only params
def example(pos_only, /, normal, *, kw_only):
    return pos_only, normal, kw_only

print(example(1, 2, kw_only=3))
print(example(1, normal=2, kw_only=3))

## 6. Numbers & Precision

- Python's `int` has **arbitrary precision** — it never overflows (limited only by memory).
- `float` is a 64-bit IEEE-754 double, so it has the classic binary floating-point rounding issues (`0.1 + 0.2 != 0.3`).
- `/` is *true* division and always returns a `float`; `//` is *floor* division.
- `bool` is a **subclass of `int`** — `True == 1` and `False == 0`.
- `complex` numbers are built in (`1 + 2j`).

In [ ]:
# Arbitrary precision integers
big = 2 ** 100
print(big)
print(type(big))

# Floating point imprecision
print(0.1 + 0.2)              # 0.30000000000000004, not exactly 0.3
print(0.1 + 0.2 == 0.3)       # False!
import math
print(math.isclose(0.1 + 0.2, 0.3))   # correct way to compare floats

# True division vs floor division
print(7 / 2)     # 3.5 (float)
print(7 // 2)    # 3   (int, floors toward negative infinity)
print(-7 // 2)   # -4, not -3 - floor division rounds DOWN, not toward zero

# bool is a subclass of int
print(isinstance(True, int))
print(True + True)     # 2
print(True == 1, False == 0)

# Complex numbers
z = 2 + 3j
print(z.real, z.imag, abs(z))

## 7. Strings

- Strings are **immutable** sequences of Unicode code points.
- String literals made of identifier-like text are often **interned** by CPython (reused as the same object) as an optimization.
- Strings are iterable and support slicing just like lists, but slicing a string always returns a new `str`.
- Multiplying/adding strings creates new objects each time — repeated concatenation in a loop is O(n²); prefer `"".join(...)`.

In [ ]:
s = "hello"
print(s[::-1])          # reverse via slicing
print(s * 3)             # "hellohellohello"

# String interning (CPython detail)
a = "hello"
b = "hello"
print(a is b)            # True - identifier-like literals are often interned

c = "".join(["h", "e", "l", "l", "o"])
print(c == a, c is a)    # equal in value, but not guaranteed to be the same object

# Inefficient vs efficient concatenation
parts = ["a", "b", "c", "d"]
# Inefficient (creates a new string each iteration):
result = ""
for p in parts:
    result += p
# Efficient:
result2 = "".join(parts)
print(result, result2)

# Strings are iterable
print(list("abc"))
print("a" in "abc")

## 8. Lists

- Lists preserve **insertion order** and allow duplicates.
- Slicing a list always returns a **new** list (a copy of that range), never a view.
- `list.sort()` sorts **in place** and returns `None`; `sorted(list)` returns a **new** sorted list.
- Assignment does **not** copy a list — `b = a` makes `b` an alias for the same list as `a`.
- List indexing out of range raises `IndexError`, but out-of-range **slicing** does not — it just clamps.

In [ ]:
a = [3, 1, 2]
b = a                 # alias, NOT a copy
b.append(4)
print(a)               # a is affected too! [3, 1, 2, 4]

c = a.copy()           # or a[:] or list(a) - a real (shallow) copy
c.append(99)
print(a, c)             # a unaffected

# sort() returns None (in-place); sorted() returns a new list
nums = [3, 1, 2]
result = nums.sort()
print(nums, result)     # [1, 2, 3] None

nums2 = [3, 1, 2]
new_list = sorted(nums2)
print(nums2, new_list)  # original untouched, new sorted list returned

# Out-of-range slicing doesn't raise, indexing does
lst = [1, 2, 3]
print(lst[10:20])       # [] - no error
try:
    print(lst[10])
except IndexError as e:
    print("IndexError:", e)

## 9. Tuples

- Tuples are ordered and immutable, but **can contain mutable objects** (see the mutability section above).
- A **single-element tuple** requires a trailing comma: `(1,)` — `(1)` is just the integer `1` in parentheses.
- Tuples are generally faster to create and iterate than lists, and are hashable **if all their elements are hashable** — so they can be used as dict keys.
- Tuple packing/unpacking works with `*` to grab "the rest".

In [ ]:
not_a_tuple = (1)
actual_tuple = (1,)
print(type(not_a_tuple), type(actual_tuple))

# Tuples as dict keys (only if all elements are hashable)
locations = {(0, 0): "origin", (1, 1): "diagonal"}
print(locations[(0, 0)])

bad_key = ([1, 2], 3)
try:
    d = {bad_key: "won't work"}
except TypeError as e:
    print("TypeError:", e)

# Extended unpacking with *
first, *middle, last = (1, 2, 3, 4, 5)
print(first, middle, last)   # 1 [2, 3, 4] 5

## 10. Sets, Dictionaries & Hashability

- Sets and dict keys must contain only **hashable** (effectively immutable) objects.
- Sets are unordered and automatically de-duplicate; membership testing (`in`) is on average **O(1)**, vs **O(n)** for a list.
- Since **Python 3.7**, regular dicts officially preserve **insertion order** (this was a CPython implementation detail in 3.6, guaranteed by the language spec from 3.7 onward).
- Dict keys must be unique — assigning to an existing key overwrites the value.

In [ ]:
# Hashability requirement
s = {1, 2, 3}
try:
    s.add([4, 5])          # lists are unhashable
except TypeError as e:
    print("TypeError:", e)

s.add((4, 5))               # tuples are fine
print(s)

# Fast membership testing
big_list = list(range(100000))
big_set = set(big_list)
import timeit
# (set lookup is dramatically faster than list lookup for large collections)
print(99999 in big_set, 99999 in big_list)

# Dicts preserve insertion order (guaranteed since Python 3.7)
d = {}
d["z"] = 1
d["a"] = 2
d["m"] = 3
print(list(d.keys()))   # ['z', 'a', 'm'] - insertion order, not sorted

# Duplicate keys: last assignment wins
d2 = {"x": 1, "x": 2}
print(d2)   # {'x': 2}

## 11. Booleans & Truthiness

- Every object has an implicit boolean value ("truthiness") used by `if`, `while`, `and`, `or`, `not`.
- **Falsy** values: `False`, `None`, `0`, `0.0`, `0j`, `""`, `()`, `[]`, `{}`, `set()`, `range(0)`, and any object whose `__bool__`/`__len__` says so.
- Everything else is **truthy**.
- `and`/`or` **short-circuit** and return one of their actual operands (not necessarily `True`/`False`).

In [ ]:
falsy_values = [False, None, 0, 0.0, 0j, "", (), [], {}, set(), range(0)]
print(all(not bool(v) for v in falsy_values))

# and/or return an operand, not necessarily a bool
print(0 or "default")     # "default" - first truthy value
print("hi" and "bye")     # "bye" - both truthy, returns the LAST one
print([] or {})            # {} - both falsy, returns the last one

# Short-circuiting avoids evaluating the second operand when unnecessary
def side_effect():
    print("called!")
    return True

print(False and side_effect())   # side_effect() never runs
print(True or side_effect())     # side_effect() never runs

## 12. Shallow vs Deep Copy

- `=` never copies — it just binds another name to the same object.
- A **shallow copy** (`.copy()`, `list(x)`, `x[:]`, `copy.copy()`) creates a new outer container, but nested mutable objects inside are still shared.
- A **deep copy** (`copy.deepcopy()`) recursively copies everything, so nested objects are fully independent.

In [ ]:
import copy

original = [[1, 2], [3, 4]]

shallow = original.copy()
shallow[0].append(99)          # mutates the shared inner list
print("original after shallow mutation:", original)   # inner list changed too!

deep = copy.deepcopy(original)
deep[0].append(-1)
print("original after deep copy mutation:", original)  # unaffected this time

## 13. Comparisons & Operators

- Comparisons can be **chained**: `a < b < c` means `a < b and b < c` (and `b` is only evaluated once).
- `+=` on a list mutates it in place (calls `__iadd__`); `x = x + [...]` creates a new list — subtle difference when the list is referenced elsewhere.
- `==` can be overridden per class via `__eq__`; without it, objects compare by identity by default.

In [ ]:
print(1 < 2 < 3)          # True - chained comparison
print(1 < 2 > 0)           # True
print(3 < 2 < 100)         # False, short-circuits after first False

# += mutates in place for lists, unlike x = x + [...]
a = [1, 2]
b = a
a += [3]          # in-place extend (__iadd__) - b sees the change
print(a, b)

c = [1, 2]
d = c
c = c + [3]        # creates a NEW list, rebinds c only - d is unaffected
print(c, d)

# Default equality is identity-based unless __eq__ is defined
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

p1 = Point(1, 2)
p2 = Point(1, 2)
print(p1 == p2)     # False - default __eq__ compares identity

class ComparablePoint:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

cp1 = ComparablePoint(1, 2)
cp2 = ComparablePoint(1, 2)
print(cp1 == cp2)   # True - custom __eq__

## 14. Iterables, Iterators & Generators

- An **iterable** has `__iter__`; an **iterator** additionally has `__next__` and remembers its position.
- Generators (functions using `yield`, or generator expressions `(x for x in ...)`) are iterators that produce values **lazily**, one at a time, using almost no memory up front.
- A generator/iterator can only be consumed **once** — once exhausted, you must create a new one.
- `range()` is a lazy, memory-efficient sequence — it doesn't build a list of all its numbers.

In [ ]:
def gen():
    yield 1
    yield 2
    yield 3

g = gen()
print(list(g))     # [1, 2, 3]
print(list(g))     # [] - already exhausted, generators are single-use!

# range() doesn't store all values, computes them on demand
r = range(10**12)          # instant, no memory blow-up
print(r[500000000000])      # O(1) direct index access

# Generator expression vs list comprehension: lazy vs eager
gen_expr = (x * x for x in range(5))    # nothing computed yet
list_comp = [x * x for x in range(5)]   # fully computed immediately
print(type(gen_expr), type(list_comp))
print(list(gen_expr))

## 15. Classes & OOP

- Python supports **duck typing**: "if it walks like a duck and quacks like a duck" — an object's suitability is determined by the presence of methods/attributes, not its declared type.
- Multiple inheritance is resolved via the **Method Resolution Order (MRO)**, computed with the C3 linearization algorithm (`ClassName.__mro__`).
- Instance attributes are stored per-instance; **class attributes are shared** across all instances unless shadowed.
- "Private" attributes (`__name`) are just **name-mangled** (`_ClassName__name`), not truly private — Python has no enforced access control.

In [ ]:
# Duck typing
class Duck:
    def speak(self): return "Quack!"
class Dog:
    def speak(self): return "Woof!"

for animal in [Duck(), Dog()]:
    print(animal.speak())   # works for both - no shared base class required

# MRO with multiple inheritance
class A:
    def greet(self): return "A"
class B(A):
    def greet(self): return "B"
class C(A):
    def greet(self): return "C"
class D(B, C):
    pass

print(D.__mro__)      # D -> B -> C -> A -> object
print(D().greet())     # "B" - follows the MRO, not just "first parent"

# Class attributes are shared across instances
class Counter:
    count = 0                # class attribute
    def __init__(self):
        Counter.count += 1    # mutating the CLASS attribute

Counter(); Counter(); Counter()
print(Counter.count)   # 3 - shared across all instances

# Name mangling for "private" attributes
class Secret:
    def __init__(self):
        self.__hidden = 42     # becomes _Secret__hidden

s = Secret()
print(s._Secret__hidden)   # still accessible - not truly private
print(hasattr(s, "__hidden"))

## 16. Miscellaneous Facts

- CPython (the reference implementation) has a **Global Interpreter Lock (GIL)**, which allows only one thread to execute Python bytecode at a time — CPU-bound multithreading doesn't get true parallelism (use `multiprocessing` instead); I/O-bound work still benefits from threads.
- The walrus operator `:=` (Python 3.8+) lets you assign a value **as part of an expression**.
- A single leading underscore (`_var`) is a naming *convention* for "internal use" — it is not enforced.
- `import` only runs a module's top-level code **once** per process; subsequent imports reuse the cached module from `sys.modules`.
- `__name__ == "__main__"` is `True` only when a file is run directly, not when it's imported.

In [ ]:
# Walrus operator - assign within an expression
data = [1, 2, 3, 4, 5]
if (n := len(data)) > 3:
    print(f"List is long: {n} items")

# while loop with walrus (common pattern for reading until a sentinel)
import io
stream = io.StringIO("line1\nline2\nline3\n")
lines = []
while (line := stream.readline()):
    lines.append(line.strip())
print(lines)

# Modules are cached - only executed once
import sys
print("math" in sys.modules)   # True if math was imported earlier in this notebook

# __name__ trick
print(__name__)   # "__main__" when run directly in a notebook/script